In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/AI Project/saket_processed.csv'
df = pd.read_csv(file_path)

In [ ]:
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20000, 21)


,order_volume,warehouse_time_hrs,shipment_distance_km,traffic_level,weather_severity,courier_load_pct,past_delay_rate,risk_score,delivery_status,day_of_week,...,warehouse_region,delivery_type,delivery_status_encoded,severity_index,warehouse_efficiency,log_order_volume,log_shipment_distance,dist_Short,dist_Medium,dist_Long
0,94,4.314205,2307.511608,2,0,45.789611,0.332918,0.486537,On-Time,6,...,West,Same-Day,0,2,534.863659,4.553877,7.744358,False,False,True
1,484,26.992300,1721.851885,5,3,36.217764,0.326364,0.745403,At Risk,3,...,Central,Express,1,20,63.790484,6.184149,7.451736,False,True,False
2,432,37.363071,1919.134927,4,0,86.955847,0.070964,0.590096,At Risk,2,...,North,Express,1,4,51.364485,6.070738,7.560151,False,True,False
3,374,2.536696,373.240260,3,1,85.850780,0.328125,0.543604,At Risk,6,...,North,Same-Day,1,6,147.136382,5.926926,5.924898,True,False,False
4,126,8.527455,1593.551859,5,3,39.788405,0.430941,0.600323,At Risk,0,...,West,Standard,1,20,186.873087,4.844187,7.374348,False,True,False


In [ ]:
df.columns

Index(['order_volume', 'warehouse_time_hrs', 'shipment_distance_km',
       'traffic_level', 'weather_severity', 'courier_load_pct',
       'past_delay_rate', 'risk_score', 'delivery_status', 'day_of_week',
       'is_peak_hour', 'warehouse_region', 'delivery_type',
       'delivery_status_encoded', 'severity_index', 'warehouse_efficiency',
       'log_order_volume', 'log_shipment_distance', 'dist_Short',
       'dist_Medium', 'dist_Long'],
      dtype='object')

In [ ]:
TARGET = "delivery_status"
X = df.drop(columns=[
    TARGET,
    "delivery_status_encoded"   # <-- THIS REMOVES LEAKAGE
])
y = df[TARGET]

In [ ]:
y.unique()

array(['On-Time', 'At Risk', 'Delayed'], dtype=object)

In [ ]:
label_map = {
    "On-Time": 0,
    "At Risk": 1,
    "Delayed": 2
}
y = y.map(label_map)
y.value_counts()

,count
delivery_status,
1,15931
0,3195
2,874


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (16000, 19)
Test size: (4000, 19)


# COLUMN PREPROCESSOR FOR SCALING AND ONE HOT ENCODING (already included in the pipeline)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

# Identify numerical and categorical columns from the original X_train
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'bool']).columns

# Create a preprocessor using ColumnTransformer
# 'num' pipeline for numerical features: apply StandardScaler
# 'cat' pipeline for categorical features: apply OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep any columns not specified (e.g., if there were others)
)

# Fit the preprocessor on the training data and transform both training and test data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Note: ColumnTransformer output is a NumPy array. If you need it back as a DataFrame
# and can handle the large number of columns from one-hot encoding, you can convert it.
# For most sklearn models, NumPy array input is perfectly fine.

print("Shape of X_train_preprocessed:", X_train_preprocessed.shape)
print("Shape of X_test_preprocessed:", X_test_preprocessed.shape)

Shape of X_train_preprocessed: (16000, 28)
Shape of X_test_preprocessed: (4000, 28)


# PIPELINES FOR 7 MODELS

In [ ]:
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Detect numeric / categorical feature columns from original X
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object", "bool"]).columns

# Shared preprocessor: scale nums, one‑hot cats
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
)

# 1) Logistic Regression
logreg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=2000,
        multi_class="multinomial",
        C=0.01,
        random_state=42,
    )),
])
logreg_pipeline.fit(X_train, y_train)
print("LogReg acc:", logreg_pipeline.score(X_test, y_test))

# 2) Random Forest
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )),
])
rf_pipeline.fit(X_train, y_train)
print("RF acc:", rf_pipeline.score(X_test, y_test))

# 3) HistGradientBoosting (tree boosting, fast on tabular)
hgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", HistGradientBoostingClassifier(
        max_iter=200,
        random_state=42,
    )),
])
hgb_pipeline.fit(X_train, y_train)
print("HGB acc:", hgb_pipeline.score(X_test, y_test))

# 4) Linear SVM
svm_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LinearSVC(
        C=1.0,
        random_state=42,
    )),
])
svm_pipeline.fit(X_train, y_train)
print("Linear SVM acc:", svm_pipeline.score(X_test, y_test))

# 5) LightGBM (tree model, no scaling needed but OK in this pipeline)
lgbm_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LGBMClassifier(
        random_state=42,
    )),
])
lgbm_pipeline.fit(X_train, y_train)
y_pred_lgbm = lgbm_pipeline.predict(X_test)
print("LightGBM acc:", accuracy_score(y_test, y_pred_lgbm))
print("LightGBM report:\n", classification_report(y_test, y_pred_lgbm))

# 6) Gradient Boosting (sklearn)
gb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", GradientBoostingClassifier(
        random_state=42,
    )),
])
gb_pipeline.fit(X_train, y_train)
print("GradientBoosting acc:", gb_pipeline.score(X_test, y_test))

# 7) XGBoost (sklearn API)
xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )),
])
xgb_pipeline.fit(X_train, y_train)
print("XGBoost acc:", xgb_pipeline.score(X_test, y_test))

# Persist models for FastAPI
joblib.dump(logreg_pipeline, "logreg_pipeline.joblib")
joblib.dump(rf_pipeline, "rf_pipeline.joblib")
joblib.dump(hgb_pipeline, "hgb_pipeline.joblib")
joblib.dump(svm_pipeline, "svm_pipeline.joblib")
joblib.dump(lgbm_pipeline, "lgbm_pipeline.joblib")
joblib.dump(gb_pipeline, "gb_pipeline.joblib")
joblib.dump(xgb_pipeline, "xgb_pipeline.joblib")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg acc: 0.896
RF acc: 0.91425
HGB acc: 0.90975
Linear SVM acc: 0.8795
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001905 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2353
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 28
[LightGBM] [Info] Start training from score -1.834145
[LightGBM] [Info] Start training from score -0.227450
[LightGBM] [Info] Start training from score -3.130693


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM acc: 0.91025
LightGBM report:
               precision    recall  f1-score   support

           0       0.78      0.95      0.86       639
           1       0.96      0.93      0.94      3186
           2       0.55      0.42      0.48       175

    accuracy                           0.91      4000
   macro avg       0.76      0.77      0.76      4000
weighted avg       0.91      0.91      0.91      4000

GradientBoosting acc: 0.9185
XGBoost acc: 0.90925


['xgb_pipeline.joblib']

In [ ]:
from google.colab import files

file_names = ["logreg_pipeline.joblib", "rf_pipeline.joblib", "hgb_pipeline.joblib", "svm_pipeline.joblib", "lgbm_pipeline.joblib", "gb_pipeline.joblib", "xgb_pipeline.joblib"]
for file_name in file_names:
    files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# INDIVIDUAL MODELS

In [ ]:
# import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        solver='lbfgs',
        max_iter=2000,
        multi_class='multinomial',
        C=0.01,
        # class_weight='balanced',
        random_state=42
    ))
])

clf.fit(X_train_preprocessed, y_train)
print(clf.score(X_test_preprocessed, y_test))
# joblib.dump(clf, "logreg_pipeline.joblib")

# param_grid = {
#     'logreg__C': [0.01, 0.1, 1.0, 10.0]
# }

# grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
# grid.fit(X_train_preprocessed, y_train)

# print("Best params:", grid.best_params_)
# print("CV score:", grid.best_score_)
# print("Test score:", grid.best_estimator_.score(X_test_preprocessed, y_test))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


0.8955


In [ ]:
from google.colab import files
files.download("logreg_pipeline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler # Keep for consistency in imports, though removed from pipeline
import joblib

# Remove the StandardScaler from the pipeline as X_train_preprocessed and X_test_preprocessed are already processed.
# Tree-based models like RandomForest also generally do not require feature scaling.
rf_clf = Pipeline([
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Fit and score using the preprocessed data
rf_clf.fit(X_train_preprocessed, y_train)
print("RF test accuracy:", rf_clf.score(X_test_preprocessed, y_test))
joblib.dump(rf_clf, "rf_pipeline.joblib")

RF test accuracy: 0.91425


['rf_pipeline.joblib']

In [ ]:
from google.colab import files
files.download("rf_pipeline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_clf = HistGradientBoostingClassifier(
    max_depth=None,
    learning_rate=0.1,
    max_iter=200,
    random_state=42
)

# no StandardScaler needed; trees don't care about scaling
hgb_clf.fit(X_train_preprocessed, y_train)
print("HGB test accuracy:", hgb_clf.score(X_test_preprocessed, y_test))

joblib.dump(hgb_clf, "hgb_model.joblib")


HGB test accuracy: 0.90975


['hgb_model.joblib']

In [ ]:
from google.colab import files
files.download("hgb_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.svm import LinearSVC

svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        random_state=42
    ))
])

svm_clf.fit(X_train_preprocessed, y_train)
print("Linear SVM test accuracy:", svm_clf.score(X_test_preprocessed, y_test))

joblib.dump(svm_clf, "svm_pipeline.joblib")


Linear SVM test accuracy: 0.8795


['svm_pipeline.joblib']

In [ ]:
from google.colab import files
files.download("svm_pipeline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## LightGBM Model Training and Evaluation

### Subtask:
Train a LightGBM classifier using `X_train` and `y_train`. Make predictions on `X_test` and evaluate its performance using accuracy and classification report. Add the accuracy to the comparison DataFrame.


In [ ]:
# !pip install lightgbm
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Re-initialize comparison DataFrame if it doesn't exist or is incomplete
# Only include Logistic Regression as Random Forest prediction was interrupted
# if 'comparison' not in locals() or comparison.shape[0] == 0:
#     comparison = pd.DataFrame({
#         "Model": ["Logistic Regression"],
#         "Accuracy": [accuracy_score(y_test, y_pred_lr)]
#     })

# Train LightGBM model and make predictions
model_lgbm = LGBMClassifier(
    random_state=42,
    class_weight="balanced"
)

model_lgbm.fit(X_train_preprocessed, y_train)

y_pred_lgbm = model_lgbm.predict(X_test_preprocessed)

# Evaluate LightGBM and update comparison DataFrame
print("LightGBM Accuracy:", accuracy_score(y_test, y_pred_lgbm))
print(classification_report(y_test, y_pred_lgbm))

# comparison = pd.concat([comparison, pd.DataFrame({
#     "Model": ["LightGBM"],
#     "Accuracy": [accuracy_score(y_test, y_pred_lgbm)]
# })], ignore_index=True)

# comparison

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002845 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2353
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 28
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
LightGBM Accuracy: 0.906
              precision    recall  f1-score   support

           0       0.78      0.99      0.87       639
           1       0.99      0.89      0.94      3186
           2       0.48      0.86      0.61       175

    accuracy                           0.91      4000
   macro avg       0.75      0.91      0.81      4000
weighted avg       0.93      0.91      0.91      4000



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## XGBoost Model Training and Evaluation

### Subtask:
Train an XGBoost classifier using `X_train` and `y_train`. Make predictions on `X_test` and evaluate its performance using accuracy and classification report. Add the accuracy to the comparison DataFrame.

In [ ]:
pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 1.3 MB/s eta 0:00:00


In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

model_xgb.fit(X_train_preprocessed, y_train)

y_pred_xgb = model_xgb.predict(X_test_preprocessed)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

# comparison = pd.concat([comparison, pd.DataFrame({
#     "Model": ["XGBoost"],
#     "Accuracy": [accuracy_score(y_test, y_pred_xgb)]
# })], ignore_index=True)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [06:01:45] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.909
              precision    recall  f1-score   support

           0       0.78      0.92      0.85       639
           1       0.95      0.93      0.94      3186
           2       0.59      0.43      0.50       175

    accuracy                           0.91      4000
   macro avg       0.78      0.76      0.76      4000
weighted avg       0.91      0.91      0.91      4000



In [ ]:
comparison = comparison.drop_duplicates(subset=['Model'], keep='first')
comparison

,Model,Accuracy
0,Logistic Regression,0.751
1,LightGBM,0.842
2,XGBoost,0.845


## Gradient Boosting Model Training and Evaluation

### Subtask:
Train a Gradient Boosting classifier using `X_train` and `y_train`. Make predictions on `X_test` and evaluate its performance using accuracy and classification report. Add the accuracy to the comparison DataFrame.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

model_gb = GradientBoostingClassifier(
    random_state=42
)

model_gb.fit(X_train_preprocessed, y_train)

y_pred_gb = model_gb.predict(X_test_preprocessed)

print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

# comparison = pd.concat([comparison, pd.DataFrame({
#     "Model": ["Gradient Boosting"],
#     "Accuracy": [accuracy_score(y_test, y_pred_gb)]
# })], ignore_index=True)

# comparison

Gradient Boosting Accuracy: 0.9185
              precision    recall  f1-score   support

           0       0.78      1.00      0.88       639
           1       0.96      0.94      0.95      3186
           2       0.73      0.28      0.40       175

    accuracy                           0.92      4000
   macro avg       0.82      0.74      0.74      4000
weighted avg       0.92      0.92      0.91      4000



## Artificial Neural Network (ANN) Model Training and Evaluation

### Subtask:
Install the TensorFlow library to enable ANN model training.


In [ ]:
pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 822.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.9 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Convert y_train and y_test to one-hot encoding for categorical cross-entropy loss
y_train_categorical = to_categorical(y_train)
y_test_categorical = to_categorical(y_test)

# Define the ANN model
model_ann = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_preprocessed.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')  # 3 output classes: On-Time, At Risk, Delayed
])

# Compile the model
model_ann.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model_ann.fit(
    X_train_preprocessed,
    y_train_categorical,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("ANN model trained successfully.")

# Make predictions on the scaled test data
y_pred_ann_probs = model_ann.predict(X_test_preprocessed)
y_pred_ann = tf.argmax(y_pred_ann_probs, axis=1)

# Evaluate ANN and update comparison DataFrame
print("ANN Accuracy:", accuracy_score(y_test, y_pred_ann))
print(classification_report(y_test, y_pred_ann))

# comparison = pd.concat([comparison, pd.DataFrame({
#     "Model": ["Artificial Neural Network"],
#     "Accuracy": [accuracy_score(y_test, y_pred_ann)]
# })], ignore_index=True)

# comparison

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ANN model trained successfully.
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
ANN Accuracy: 0.88475
              precision    recall  f1-score   support

           0       0.75      0.80      0.77       639
           1       0.93      0.93      0.93      3186
           2       0.54      0.42      0.47       175

    accuracy                           0.88      4000
   macro avg       0.74      0.72      0.73      4000
weighted avg       0.88      0.88      0.88      4000



In [ ]:
print("--- Model Performance Summary ---")
print(comparison.sort_values(by='Accuracy', ascending=False))

best_model = comparison.sort_values(by='Accuracy', ascending=False).iloc[0]
print(f"\nRecommendation: The best-performing model is {best_model['Model']} with an accuracy of {best_model['Accuracy']:.3f}.")

--- Model Performance Summary ---
                       Model  Accuracy
3          Gradient Boosting     0.857
2                    XGBoost     0.845
1                   LightGBM     0.842
4  Artificial Neural Network     0.834
0        Logistic Regression     0.751

Recommendation: The best-performing model is Gradient Boosting with an accuracy of 0.857.


# IGNORE BELOW THIS FOR NOW

## Hyperparameter Tuning for Gradient Boosting

### Subtask:
Perform GridSearchCV to find optimal hyperparameters for the Gradient Boosting Classifier using `X_train` and `y_train`. Evaluate the tuned model's performance on `X_test` using accuracy and classification report, and update the `comparison` DataFrame with the new accuracy.


**Reasoning**:
To begin the hyperparameter tuning process, I need to import the necessary modules, `GridSearchCV` and `GradientBoostingClassifier`, and then define the range of hyperparameters to search through for the Gradient Boosting model. Given that `X_train_scaled` and `X_test_scaled` were used for the ANN model, I will ensure to use `X_train` and `X_test` as per the problem description for consistency with other tree-based models.



In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# Define the parameter grid for GridSearchCV
param_grid_gb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9]
}

print("Defined parameter grid for Gradient Boosting.")

Defined parameter grid for Gradient Boosting.


**Reasoning**:
Now that the parameter grid is defined, I will initialize `GridSearchCV` with `GradientBoostingClassifier` and the defined parameter grid. Then, I will fit it to `X_train` and `y_train`, print the best parameters, make predictions on `X_test` using the best estimator, evaluate its performance, and update the `comparison` DataFrame. I will also remove any previous 'Gradient Boosting' entries from the `comparison` DataFrame to ensure only the tuned model's accuracy is present.



In [ ]:
grid_search_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid_gb,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for Gradient Boosting...")
grid_search_gb.fit(X_train_preprocessed, y_train)

print(f"Best parameters for Gradient Boosting: {grid_search_gb.best_params_}")

best_model_gb = grid_search_gb.best_estimator_
y_pred_gb_tuned = best_model_gb.predict(X_test_preprocessed)

print("Tuned Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb_tuned))
print(classification_report(y_test, y_pred_gb_tuned))

# Remove existing 'Gradient Boosting' entries if any, then add the tuned one
# comparison = comparison[comparison['Model'] != 'Gradient Boosting']
# comparison = pd.concat([comparison, pd.DataFrame({
#     "Model": ["Gradient Boosting (Tuned)"],
#     "Accuracy": [accuracy_score(y_test, y_pred_gb_tuned)]
# })], ignore_index=True)

# comparison

Starting GridSearchCV for Gradient Boosting...
Fitting 3 folds for each of 81 candidates, totalling 243 fits


KeyboardInterrupt: 

## Hyperparameter Tuning for XGBoost

### Subtask:
Perform GridSearchCV to find optimal hyperparameters for the XGBoost Classifier using `X_train` and `y_train`. Evaluate the tuned model's performance on `X_test` using accuracy and classification report, and update the `comparison` DataFrame with the new accuracy.

**Reasoning**:
To begin hyperparameter tuning for XGBoost, I need to define a parameter grid for `XGBClassifier` that `GridSearchCV` will explore. This includes parameters like `n_estimators`, `learning_rate`, `max_depth`, and `subsample` to find the optimal combination.



In [ ]:
from xgboost import XGBClassifier

# Define the parameter grid for GridSearchCV for XGBoost
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9]
}

print("Defined parameter grid for XGBoost.")

Defined parameter grid for XGBoost.


**Reasoning**:
Now that the parameter grid for XGBoost is defined, I will initialize `GridSearchCV` with `XGBClassifier` and the defined parameter grid. Then, I will fit it to `X_train` and `y_train`, print the best parameters, make predictions on `X_test` using the best estimator, evaluate its performance, and update the `comparison` DataFrame. I will also remove any previous 'XGBoost' entries from the `comparison` DataFrame to ensure only the tuned model's accuracy is present.



In [ ]:
grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    param_grid=param_grid_xgb,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for XGBoost...")
grid_search_xgb.fit(X_train_preprocessed, y_train)

print(f"Best parameters for XGBoost: {grid_search_xgb.best_params_}")

best_model_xgb = grid_search_xgb.best_estimator_
y_pred_xgb_tuned = best_model_xgb.predict(X_test_preprocessed)

print("Tuned XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb_tuned))
print(classification_report(y_test, y_pred_xgb_tuned))

# Remove existing 'XGBoost' entries if any, then add the tuned one
# comparison = comparison[comparison['Model'] != 'XGBoost']
# comparison = pd.concat([comparison, pd.DataFrame({
#     "Model": ["XGBoost (Tuned)"],
#     "Accuracy": [accuracy_score(y_test, y_pred_xgb_tuned)]
# })], ignore_index=True)

# comparison

Starting GridSearchCV for XGBoost...
Fitting 3 folds for each of 81 candidates, totalling 243 fits


KeyboardInterrupt: 

**Reasoning**:
The subtask for XGBoost tuning is complete. The next logical step is to proceed with hyperparameter tuning for the LightGBM model, following the main task's instructions.



**Reasoning**:
To begin hyperparameter tuning for LightGBM, I need to define a parameter grid for `LGBMClassifier` that `GridSearchCV` will explore. This includes parameters like `n_estimators`, `learning_rate`, `max_depth`, and `subsample` to find the optimal combination. Since `X_train` and `y_train` are already prepared, I will use them for training.



In [ ]:
from lightgbm import LGBMClassifier

# Define the parameter grid for GridSearchCV for LightGBM
param_grid_lgbm = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9]
}

print("Defined parameter grid for LightGBM.")

**Reasoning**:
Now that the parameter grid for LightGBM is defined, I will initialize `GridSearchCV` with `LGBMClassifier` and the defined parameter grid. Then, I will fit it to `X_train` and `y_train`, print the best parameters, make predictions on `X_test` using the best estimator, evaluate its performance, and update the `comparison` DataFrame. I will also remove any previous 'LightGBM' entries from the `comparison` DataFrame to ensure only the tuned model's accuracy is present.



In [ ]:
grid_search_lgbm = GridSearchCV(
    estimator=LGBMClassifier(random_state=42, class_weight="balanced"),
    param_grid=param_grid_lgbm,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for LightGBM...")
grid_search_lgbm.fit(X_train_preprocessed, y_train)

print(f"Best parameters for LightGBM: {grid_search_lgbm.best_params_}")

best_model_lgbm = grid_search_lgbm.best_estimator_
y_pred_lgbm_tuned = best_model_lgbm.predict(X_test_preprocessed)

print("Tuned LightGBM Accuracy:", accuracy_score(y_test, y_pred_lgbm_tuned))
print(classification_report(y_test, y_pred_lgbm_tuned))

# Remove existing 'LightGBM' entries if any, then add the tuned one
# comparison = comparison[comparison['Model'] != 'LightGBM']
# comparison = pd.concat([
#     comparison,
#     pd.DataFrame({
#         "Model": ["LightGBM (Tuned)"],
#         "Accuracy": [accuracy_score(y_test, y_pred_lgbm_tuned)]
#     })
# ], ignore_index=True)

# comparison

## Hyperparameter Retuning for Artificial Neural Network (ANN)

### Subtask:
Retune the Artificial Neural Network (ANN) model using `X_train_scaled` and `y_train_categorical`. Evaluate the retuned model's performance on `X_test_scaled` using accuracy and classification report, and update the `comparison` DataFrame with the new accuracy.

**Reasoning**:
To retune the ANN model, I will define a new model architecture, compile it, and train it using the scaled training data and categorical labels. After training, I will make predictions on the scaled test data, evaluate its performance using accuracy and a classification report, and then update the `comparison` DataFrame, removing any previous ANN entries to ensure only the retuned model's performance is reflected.



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Define the retuned ANN model (e.g., slightly different architecture or more epochs)
model_ann_retuned = Sequential([
    Dense(164, activation='relu', input_shape=(X_train_scaled.shape[1],)), # Increased units
    Dense(82, activation='relu'), # Increased units
    Dense(3, activation='softmax')  # 3 output classes: On-Time, At Risk, Delayed
])

# Compile the model
model_ann_retuned.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting ANN Retuning...")

# Train the model with potentially more epochs
history_retuned = model_ann_retuned.fit(
    X_train_scaled,
    y_train_categorical,
    epochs=75, # Increased epochs
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Retuned ANN model trained successfully.")

# Make predictions on the scaled test data
y_pred_ann_retuned_probs = model_ann_retuned.predict(X_test_scaled)
y_pred_ann_retuned = tf.argmax(y_pred_ann_retuned_probs, axis=1)

# Evaluate retuned ANN and update comparison DataFrame
print("Retuned ANN Accuracy:", accuracy_score(y_test, y_pred_ann_retuned))
print(classification_report(y_test, y_pred_ann_retuned))

# Remove existing 'Artificial Neural Network' entries if any, then add the retuned one
comparison = comparison[comparison['Model'] != 'Artificial Neural Network']
comparison = pd.concat([comparison, pd.DataFrame({
    "Model": ["Artificial Neural Network (Retuned)"],
    "Accuracy": [accuracy_score(y_test, y_pred_ann_retuned)]
})], ignore_index=True)

comparison

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting ANN Retuning...
Retuned ANN model trained successfully.
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Retuned ANN Accuracy: 0.806
              precision    recall  f1-score   support

           0       0.85      0.81      0.83       428
           1       0.82      0.82      0.82       537
           2       0.34      0.46      0.39        35

    accuracy                           0.81      1000
   macro avg       0.67      0.70      0.68      1000
weighted avg       0.81      0.81      0.81      1000



,Model,Accuracy
0,Logistic Regression,0.751
1,Gradient Boosting (Tuned),0.869
2,XGBoost (Tuned),0.871
3,LightGBM (Tuned),0.849
4,Artificial Neural Network (Retuned),0.806


**Reasoning**:
The previous code generated a UserWarning regarding the `input_shape` argument in the `Dense` layer. To resolve this, I will modify the ANN model definition to use an `Input` layer as the first layer in the `Sequential` model, which is the recommended practice by Keras.



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Define the retuned ANN model (e.g., slightly different architecture or more epochs)
model_ann_retuned = Sequential([
    Input(shape=(X_train_scaled.shape[1],)), # Use Input layer instead of input_shape in Dense
    Dense(164, activation='relu'), # Increased units
    Dense(82, activation='relu'), # Increased units
    Dense(3, activation='softmax')  # 3 output classes: On-Time, At Risk, Delayed
])

# Compile the model
model_ann_retuned.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting ANN Retuning...")

# Train the model with potentially more epochs
history_retuned = model_ann_retuned.fit(
    X_train_scaled,
    y_train_categorical,
    epochs=75, # Increased epochs
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Retuned ANN model trained successfully.")

# Make predictions on the scaled test data
y_pred_ann_retuned_probs = model_ann_retuned.predict(X_test_scaled)
y_pred_ann_retuned = tf.argmax(y_pred_ann_retuned_probs, axis=1)

# Evaluate retuned ANN and update comparison DataFrame
print("Retuned ANN Accuracy:", accuracy_score(y_test, y_pred_ann_retuned))
print(classification_report(y_test, y_pred_ann_retuned))

# Remove existing 'Artificial Neural Network' entries if any, then add the retuned one
comparison = comparison[comparison['Model'] != 'Artificial Neural Network']
comparison = pd.concat([
    comparison,
    pd.DataFrame({
        "Model": ["Artificial Neural Network (Retuned)"],
        "Accuracy": [accuracy_score(y_test, y_pred_ann_retuned)]
    })
], ignore_index=True)

comparison

Starting ANN Retuning...
Retuned ANN model trained successfully.
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Retuned ANN Accuracy: 0.809
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       428
           1       0.83      0.81      0.82       537
           2       0.39      0.37      0.38        35

    accuracy                           0.81      1000
   macro avg       0.68      0.68      0.68      1000
weighted avg       0.81      0.81      0.81      1000



,Model,Accuracy
0,Logistic Regression,0.751
1,Gradient Boosting (Tuned),0.869
2,XGBoost (Tuned),0.871
3,LightGBM (Tuned),0.849
4,Artificial Neural Network (Retuned),0.806
5,Artificial Neural Network (Retuned),0.809


**Reasoning**:
All models have been tuned and evaluated. The next step is to display the complete `comparison` DataFrame, summarize the performance of all models (including Logistic Regression and Random Forest if available in `comparison`), and recommend the best-performing model with respect to a target accuracy of 95%.



In [ ]:
print("--- Model Performance Summary ---")
print(comparison.sort_values(by='Accuracy', ascending=False))

best_model = comparison.sort_values(by='Accuracy', ascending=False).iloc[0]
print(f"\nRecommendation: The best-performing model is {best_model['Model']} with an accuracy of {best_model['Accuracy']:.3f}.")

# Check if the best model meets the target accuracy
target_accuracy = 0.95
if best_model['Accuracy'] >= target_accuracy:
    print(f"The best model ({best_model['Model']}) meets the target accuracy of {target_accuracy:.0%}.")
else:
    print(f"The best model ({best_model['Model']}) does not meet the target accuracy of {target_accuracy:.0%}. Further improvement is needed.")

--- Model Performance Summary ---
                                 Model  Accuracy
2                      XGBoost (Tuned)     0.871
1            Gradient Boosting (Tuned)     0.869
3                     LightGBM (Tuned)     0.849
5  Artificial Neural Network (Retuned)     0.809
4  Artificial Neural Network (Retuned)     0.806
0                  Logistic Regression     0.751

Recommendation: The best-performing model is XGBoost (Tuned) with an accuracy of 0.871.
The best model (XGBoost (Tuned)) does not meet the target accuracy of 95%. Further improvement is needed.


## Final Task

### Subtask:
Summarize the performance of all models and recommend the best-performing one based on the evaluation metrics, specifically addressing whether the target accuracy of 95% was achieved.


## Summary:

### Q&A
The performance of all models was summarized, and the best-performing model was identified. The target accuracy of 95% was not achieved by any of the models.

### Data Analysis Key Findings
*   **XGBoost (Tuned)** achieved the highest accuracy among all models at 87.1%, with optimal hyperparameters of `learning_rate`: 0.01, `max_depth`: 3, `n_estimators`: 100, and `subsample`: 0.8.
*   **Gradient Boosting (Tuned)** performed very closely to XGBoost, reaching an accuracy of 86.9%. Its best parameters were `learning_rate`: 0.01, `max_depth`: 3, `n_estimators`: 100, and `subsample`: 0.8.
*   **LightGBM (Tuned)** showed an accuracy of 84.9%, with best parameters including `learning_rate`: 0.1, `max_depth`: 5, `n_estimators`: 100, and `subsample`: 0.7.
*   The **Artificial Neural Network (Retuned)** model achieved an accuracy of 80.9% after increasing layer units and epochs.
*   **Logistic Regression**, an untuned baseline model, had the lowest accuracy at 75.1%.
*   None of the models, including the best-performing XGBoost, met the target accuracy of 95%.

### Insights or Next Steps
*   Further hyperparameter tuning with a wider search space or more advanced optimization techniques could potentially improve model performance, especially for the top-performing tree-based models.
*   Consider exploring advanced feature engineering, creating new features, or gathering more relevant data to help all models achieve the ambitious 95% accuracy target.
